# Temperature and Sampling Parameters

**Estimated time:** 5-10 minutes

---

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand what temperature does and how it affects LLM outputs
- Learn about sampling parameters (top_p, top_k)
- Know when to use low vs high temperature
- Be able to control creativity vs consistency in LLM responses

## 🌡️ What is Temperature?

When an LLM generates text, it predicts the **probability** of each possible next word. Temperature controls how those probabilities are used:

### The Analogy
Think of temperature like controlling randomness in decision-making:
- **Low temperature (0.0)**: Like a cautious expert who always picks the safest, most likely answer
- **High temperature (2.0)**: Like a creative brainstormer who takes risks and explores unusual ideas

### Technical Details

```python
# Simplified version of what happens:
probabilities = softmax(logits / temperature)
```

- **Temperature = 0.0**: Deterministic (always picks highest probability word)
- **Temperature = 0.7**: Slight randomness (default for many applications)
- **Temperature = 1.0**: Full probability distribution
- **Temperature = 2.0**: Very random, creative outputs

### Visual Representation

Imagine the model's probability distribution for the next word:

```
Low Temperature (0.1):          High Temperature (2.0):
"cat":     ████████████ 90%    "cat":     ████ 40%
"dog":     ██ 8%                "dog":     ███ 25%
"bird":    █ 2%                 "bird":    ███ 20%
                                 "fish":    ██ 15%
```

Low temperature sharpens the distribution → more predictable  
High temperature flattens the distribution → more diverse

## 🔧 Setup

We'll use the Transformers library with a small model for quick demonstrations.

In [ ]:
# Install required packages if needed
# !pip install transformers torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful")

In [ ]:
# Load a small, fast model for demonstrations
# Using GPT-2 small (124M parameters) for speed
model_name = "gpt2"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print("✓ Model loaded successfully!")

In [ ]:
# Helper function to generate text with different parameters
def generate_text(prompt, temperature=1.0, top_p=1.0, top_k=50, max_length=50, num_return_sequences=1):
    """
    Generate text with specified sampling parameters.
    
    Args:
        prompt: Input text
        temperature: Controls randomness (0.0 = deterministic, 2.0 = very random)
        top_p: Nucleus sampling - only sample from top tokens with cumulative probability p
        top_k: Only sample from top k tokens
        max_length: Maximum length of generated text
        num_return_sequences: Number of different outputs to generate
    """
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Generate with specified parameters
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature if temperature > 0 else 1.0,  # Avoid division by zero
            top_p=top_p,
            top_k=top_k,
            do_sample=temperature > 0,  # Deterministic if temp=0
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode outputs
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    
    return generated_texts

print("✓ Helper functions ready")

## 🧪 Experiment 1: Temperature Comparison

Let's see how temperature affects the same prompt with different values.

In [ ]:
# Prompt for experimentation
prompt = "The future of artificial intelligence is"

# Temperature values to test
temperatures = [0.0, 0.3, 0.7, 1.0, 1.5, 2.0]

print(f"🎯 Prompt: '{prompt}'\n")
print("=" * 80)

results = []

for temp in temperatures:
    # Generate 3 outputs for each temperature to show variability
    outputs = generate_text(prompt, temperature=temp, max_length=40, num_return_sequences=3)
    
    for i, output in enumerate(outputs):
        # Remove the prompt from the output for clarity
        completion = output[len(prompt):].strip()
        results.append({
            'Temperature': temp,
            'Run': i + 1,
            'Completion': completion
        })

# Display as DataFrame
df = pd.DataFrame(results)
print("\n📊 Results:\n")
display(df)

### 🔍 What to Notice:

1. **Temperature 0.0**: All 3 runs should be identical or very similar
2. **Temperature 0.3-0.7**: Slight variations, but still coherent
3. **Temperature 1.0-1.5**: More diverse outputs
4. **Temperature 2.0**: Very creative, possibly incoherent

Try running the cell above multiple times - low temperatures give consistent results!

## 🎨 Experiment 2: Factual vs Creative Tasks

Different tasks benefit from different temperature settings.

In [ ]:
# Factual task - we want consistency
factual_prompt = "The capital of France is"

print("📚 FACTUAL TASK (want low temperature)")
print(f"Prompt: '{factual_prompt}'\n")

for temp in [0.0, 0.7, 1.5]:
    outputs = generate_text(factual_prompt, temperature=temp, max_length=20, num_return_sequences=3)
    print(f"\nTemperature {temp}:")
    for i, output in enumerate(outputs, 1):
        print(f"  {i}. {output}")

In [ ]:
# Creative task - we want diversity
creative_prompt = "Once upon a time in a magical forest,"

print("🎨 CREATIVE TASK (want higher temperature)")
print(f"Prompt: '{creative_prompt}'\n")

for temp in [0.0, 0.7, 1.5]:
    outputs = generate_text(creative_prompt, temperature=temp, max_length=30, num_return_sequences=3)
    print(f"\nTemperature {temp}:")
    for i, output in enumerate(outputs, 1):
        print(f"  {i}. {output}")

### 💡 Key Insight:

- **Factual tasks**: Use low temperature (0.0-0.3) for consistent, reliable answers
- **Creative tasks**: Use higher temperature (0.7-1.5) for diverse, imaginative outputs

## 🎲 Advanced Sampling: top_p and top_k

Temperature isn't the only way to control sampling. Let's explore two other important parameters:

### Top-k Sampling
Only consider the top k most likely tokens at each step.

```
All tokens:              Top-k (k=3):
"cat":     40%           "cat":     40%
"dog":     30%           "dog":     30%
"bird":    20%           "bird":    20%
"fish":    5%            (ignored)
"snake":   3%            (ignored)
"lizard":  2%            (ignored)
```

### Top-p (Nucleus) Sampling
Only consider tokens whose cumulative probability reaches p.

```
All tokens:              Top-p (p=0.9):
"cat":     40%           "cat":     40%
"dog":     30%           "dog":     30%
"bird":    20%           "bird":    20%
"fish":    5%            (ignored - already at 90%)
"snake":   3%            (ignored)
"lizard":  2%            (ignored)
```

Top-p is more adaptive - it includes more tokens when the model is uncertain, fewer when it's confident.

In [ ]:
# Compare different sampling strategies
prompt = "My favorite programming language is"

print(f"🎯 Prompt: '{prompt}'\n")
print("=" * 80)

# Different configurations
configs = [
    {"name": "Default", "temp": 1.0, "top_p": 1.0, "top_k": 50},
    {"name": "Top-k=10", "temp": 1.0, "top_p": 1.0, "top_k": 10},
    {"name": "Top-p=0.5", "temp": 1.0, "top_p": 0.5, "top_k": 50},
    {"name": "Top-p=0.9", "temp": 1.0, "top_p": 0.9, "top_k": 50},
    {"name": "Low temp + Top-p=0.9", "temp": 0.5, "top_p": 0.9, "top_k": 50},
]

for config in configs:
    outputs = generate_text(
        prompt, 
        temperature=config["temp"],
        top_p=config["top_p"],
        top_k=config["top_k"],
        max_length=25,
        num_return_sequences=3
    )
    
    print(f"\n{config['name']}:")
    print(f"  (temp={config['temp']}, top_p={config['top_p']}, top_k={config['top_k']})")
    for i, output in enumerate(outputs, 1):
        print(f"  {i}. {output}")

## 📊 Visual Comparison Table

Let's create a comprehensive comparison with different prompts and parameters.

In [ ]:
# Create a comparison table
test_prompts = [
    "The best way to learn programming is",
    "In the year 2050, technology will",
    "The meaning of life is"
]

temp_values = [0.0, 0.7, 1.5]

comparison_results = []

for prompt in test_prompts:
    for temp in temp_values:
        output = generate_text(prompt, temperature=temp, max_length=35, num_return_sequences=1)[0]
        comparison_results.append({
            'Prompt': prompt[:30] + "...",
            'Temperature': temp,
            'Output': output
        })

comparison_df = pd.DataFrame(comparison_results)

# Display with styling
print("\n📋 Comprehensive Comparison\n")
display(comparison_df.pivot(index='Prompt', columns='Temperature', values='Output'))

## 📚 Practical Guidelines

### When to Use Different Temperature Settings:

| Temperature | Use Case | Examples |
|-------------|----------|----------|
| **0.0 - 0.3** | Factual, deterministic tasks | - Code generation<br>- Data extraction<br>- Classification<br>- Summarization |
| **0.3 - 0.7** | Balanced tasks | - Chatbots<br>- Q&A systems<br>- General assistance |
| **0.7 - 1.2** | Creative tasks | - Story writing<br>- Brainstorming<br>- Marketing copy |
| **1.2 - 2.0** | Maximum creativity | - Poetry<br>- Experimental content<br>- Diverse ideas |

### Common Combinations:

```python
# Production chatbot (reliable but not robotic)
temperature=0.7, top_p=0.9, top_k=50

# Code generation (deterministic)
temperature=0.0, top_p=1.0, top_k=50

# Creative writing (diverse)
temperature=1.0, top_p=0.95, top_k=50

# Factual Q&A (consistent)
temperature=0.2, top_p=0.9, top_k=40
```

## 🧑‍🔬 Student Exercises

Now it's your turn! Experiment with the following exercises to deepen your understanding.

### Exercise 1: Find the Right Temperature

For each prompt below, determine the ideal temperature setting and explain why.

Test your hypotheses by running the code!

In [ ]:
# Exercise 1: Test different scenarios

scenarios = [
    {
        "task": "Extract the email address",
        "prompt": "From this text: 'Contact us at support@example.com', the email is",
        "your_temperature": 0.0,  # TODO: Choose appropriate temperature
        "reason": "Factual extraction - needs to be exact"  # TODO: Explain your choice
    },
    {
        "task": "Generate product names",
        "prompt": "Creative names for a new eco-friendly water bottle:",
        "your_temperature": 1.0,  # TODO: Choose appropriate temperature
        "reason": "Creative brainstorming"  # TODO: Explain your choice
    },
    {
        "task": "Answer science question",
        "prompt": "The process of photosynthesis involves",
        "your_temperature": 0.3,  # TODO: Choose appropriate temperature
        "reason": "Factual but may need some flexibility"  # TODO: Explain your choice
    },
]

# Test your choices
for scenario in scenarios:
    print(f"\n{'='*80}")
    print(f"Task: {scenario['task']}")
    print(f"Your temperature: {scenario['your_temperature']}")
    print(f"Reason: {scenario['reason']}")
    print(f"\nPrompt: {scenario['prompt']}\n")
    
    outputs = generate_text(
        scenario['prompt'],
        temperature=scenario['your_temperature'],
        max_length=35,
        num_return_sequences=3
    )
    
    for i, output in enumerate(outputs, 1):
        print(f"{i}. {output}")

### Exercise 2: Explore top_p

Compare outputs with different top_p values. What do you notice?

In [ ]:
# Exercise 2: Experiment with top_p

your_prompt = "The most interesting thing about space is"  # TODO: Try your own prompt!

top_p_values = [0.3, 0.6, 0.9, 1.0]

print(f"Prompt: '{your_prompt}'\n")

for top_p in top_p_values:
    print(f"\ntop_p = {top_p}:")
    outputs = generate_text(
        your_prompt,
        temperature=0.8,  # Keep temperature constant
        top_p=top_p,
        max_length=35,
        num_return_sequences=3
    )
    for i, output in enumerate(outputs, 1):
        print(f"  {i}. {output}")

# TODO: Write your observations here:
print("\n" + "="*80)
print("Your observations:")
print("- Lower top_p values (0.3):")
print("- Higher top_p values (1.0):")

### Exercise 3: Design Your Own Experiment

Create your own experiment to test a hypothesis about temperature or sampling parameters.

In [ ]:
# Exercise 3: Your custom experiment

# Example hypothesis: "Lower temperature produces shorter responses"
# TODO: Write your own hypothesis

your_hypothesis = "Write your hypothesis here"

print(f"Hypothesis: {your_hypothesis}\n")
print("="*80)

# TODO: Design and run your experiment
# Tips:
# - Choose an appropriate prompt
# - Select parameter ranges to test
# - Run multiple iterations
# - Analyze the results

# Your experiment code here:


# Your conclusions:
print("\nConclusions:")
print("Write your findings here...")

### Exercise 4: Real-World Application

Choose a real-world application and determine the optimal parameters.

In [ ]:
# Exercise 4: Real-world scenarios

applications = {
    "Customer Support Bot": {
        "prompt": "Customer asked: How do I reset my password?",
        "temperature": 0.5,  # TODO: Adjust this
        "top_p": 0.9,       # TODO: Adjust this
        "top_k": 50,        # TODO: Adjust this
        "reasoning": "Needs to be helpful but consistent"  # TODO: Explain
    },
    "Creative Story Generator": {
        "prompt": "Once upon a time, there was a dragon who",
        "temperature": 1.2,  # TODO: Adjust this
        "top_p": 0.95,      # TODO: Adjust this
        "top_k": 50,        # TODO: Adjust this
        "reasoning": "Wants maximum creativity"  # TODO: Explain
    },
    "Code Documentation Generator": {
        "prompt": "This function calculates the factorial of a number. Description:",
        "temperature": 0.2,  # TODO: Adjust this
        "top_p": 0.9,       # TODO: Adjust this
        "top_k": 40,        # TODO: Adjust this
        "reasoning": "Must be accurate and clear"  # TODO: Explain
    },
}

# Test your configurations
for app_name, config in applications.items():
    print(f"\n{'='*80}")
    print(f"Application: {app_name}")
    print(f"Parameters: temp={config['temperature']}, top_p={config['top_p']}, top_k={config['top_k']}")
    print(f"Reasoning: {config['reasoning']}")
    print(f"\nPrompt: {config['prompt']}\n")
    
    outputs = generate_text(
        config['prompt'],
        temperature=config['temperature'],
        top_p=config['top_p'],
        top_k=config['top_k'],
        max_length=40,
        num_return_sequences=2
    )
    
    for i, output in enumerate(outputs, 1):
        print(f"{i}. {output}")
    
    print(f"\nDo these outputs match your expectations? Adjust parameters if needed!")

## 🎓 Key Takeaways

### The "Aha!" Moments:

1. **Temperature is a creativity dial** 🎨
   - Low = Safe, predictable, factual
   - High = Risky, creative, diverse

2. **Different tasks need different settings** 🎯
   - Code generation? Temperature ≈ 0.0
   - Story writing? Temperature ≈ 1.0
   - Chatbot? Temperature ≈ 0.7

3. **top_p is often better than top_k** 💡
   - It adapts to the model's confidence
   - More tokens when uncertain, fewer when confident

4. **Parameters work together** 🤝
   - Combine temperature, top_p, and top_k for fine control
   - Start with defaults, then tune based on results

5. **Always test with real examples** 🧪
   - Theory is great, but experiments reveal the truth
   - Run multiple times to see consistency vs. variance

### Quick Reference:

```python
# Conservative (factual tasks)
temperature=0.2, top_p=0.9, top_k=40

# Balanced (general use)
temperature=0.7, top_p=0.9, top_k=50

# Creative (brainstorming)
temperature=1.2, top_p=0.95, top_k=50
```

## 🚀 Further Exploration

Want to learn more? Try these:

1. **Experiment with other models**: Try larger models or different architectures
2. **Measure diversity**: Calculate metrics like distinct n-grams across outputs
3. **Test repetition penalty**: Some APIs offer parameters to reduce repetitive text
4. **Compare APIs**: How do OpenAI, Anthropic, and others implement temperature?
5. **Visualize probabilities**: Plot the actual probability distributions at different temperatures

### Additional Resources:
- [Hugging Face Text Generation Guide](https://huggingface.co/docs/transformers/main_classes/text_generation)
- [The Curious Case of Neural Text Degeneration (Nucleus Sampling Paper)](https://arxiv.org/abs/1904.09751)
- [OpenAI API Documentation on Temperature](https://platform.openai.com/docs/api-reference/chat/create#chat-create-temperature)

## 💭 Reflection Questions

Before you finish, take a moment to reflect:

1. What surprised you most about temperature's effect on outputs?
2. Can you think of a use case where you'd want temperature > 2.0?
3. How might you use these parameters in a project you're working on?
4. What happens if you set both temperature and top_p very low?

---

**Congratulations!** 🎉 You now understand how to control LLM creativity and consistency using temperature and sampling parameters. This is a crucial skill for building effective AI applications!